<a href="https://colab.research.google.com/github/anitejpadyana/BIS-lab/blob/main/Gene_Expression_Programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import numpy as np
import random

N = 50
h = 5
MaxGen = 100
Pm = 0.1
Pt = 0.1
Pc = 0.7

F = ['+', '-', '*', '/']
T = ['x', 'y', '2', '5', '10', '20', '50']

n_max = 2
t_len = h * (n_max - 1) + 1
chrom_length = h + t_len

np.random.seed(42)
random.seed(42)

# Sample electricity consumption data
x_data = np.array([100, 150, 200, 250, 300, 350, 400, 450,
                   500, 550, 600, 650, 700, 750, 800, 850,
                   900, 950, 1000, 1100])

y_data = np.array([2, 3, 4, 3, 5, 4, 5, 6, 4, 5,
                   6, 5, 7, 6, 8, 7, 8, 9, 8, 10])

actual_bills = 100 + 5*x_data + 20*y_data

def create_chromosome():
    head = [random.choice(F + T) for _ in range(h)]
    tail = [random.choice(T) for _ in range(t_len)]
    return head + tail

def decode(chrom):
    index = [0]

    def build():
        if index[0] >= len(chrom):
            return ('2',)

        gene = chrom[index[0]]
        index[0] += 1

        if gene in F:
            return (gene, build(), build())

        return (gene,)

    return build()

def evaluate(tree, x, y):
    op = tree[0]

    if op == 'x':
        return x
    if op == 'y':
        return y
    if op not in F:
        return float(op)

    a = evaluate(tree[1], x, y)
    b = evaluate(tree[2], x, y)

    with np.errstate(all='ignore'):
        if op == '+':
            result = a + b
        elif op == '-':
            result = a - b
        elif op == '*':
            result = a * b
        else:
            result = np.where(np.abs(b) < 1e-6, a,
                              a / np.where(np.abs(b) < 1e-6, 1, b))

    return np.nan_to_num(result, nan=1e6, posinf=1e6, neginf=-1e6)

def expression(tree):
    op = tree[0]

    if op not in F:
        return op

    return f"({expression(tree[1])} {op} {expression(tree[2])})"

def fitness(chrom):
    tree = decode(chrom)
    prediction = evaluate(tree, x_data, y_data)

    prediction = np.clip(prediction, -1e6, 1e6)

    return np.mean((actual_bills - prediction)**2)

def select(population, scores):
    candidates = random.sample(range(N), 3)
    winner = min(candidates, key=lambda i: scores[i])
    return population[winner].copy()

def mutate(chrom):
    chrom = chrom.copy()

    for i in range(chrom_length):
        if random.random() < Pm:
            if i < h:
                chrom[i] = random.choice(F + T)
            else:
                chrom[i] = random.choice(T)

    return chrom

def transpose(chrom):
    chrom = chrom.copy()

    if random.random() < Pt:
        start = random.randint(0, h-1)
        length = random.randint(1, min(3, h-start))
        segment = chrom[start:start+length]

        position = random.randint(1, h-1)

        new_head = chrom[:h].copy()
        new_head[position:] = (segment + new_head[position:])[:h-position]

        chrom[:h] = new_head

    return chrom

def crossover(p1, p2):
    child = p1.copy()

    if random.random() < 0.5:
        point = random.randint(1, h-1)
        child[:h] = p1[:point] + p2[point:h]
    else:
        point = random.randint(1, t_len-1)
        child[h:] = p1[h:h+point] + p2[h+point:]

    return child

# Initialize population
population = [create_chromosome() for _ in range(N)]

best_chromosome = None
best_score = float('inf')

# Main GEP algorithm
for gen in range(MaxGen):

    scores = [fitness(chrom) for chrom in population]

    best_index = np.argmin(scores)

    if scores[best_index] < best_score:
        best_score = scores[best_index]
        best_chromosome = population[best_index].copy()

    new_population = [best_chromosome.copy()]

    while len(new_population) < N:
        p = select(population, scores)

        p = mutate(p)
        p = transpose(p)

        if random.random() < Pc:
            q = select(population, scores)
            p = crossover(p, q)

        new_population.append(p)

    population = new_population

# Display best solution
best_tree = decode(best_chromosome)

print("GENE EXPRESSION ALGORITHM")
print("--------------------------------")
print("Best Expression:", expression(best_tree))
print("Best Fitness (MSE):", round(best_score, 2))

print("\nELECTRICITY BILL PREDICTIONS")
print("--------------------------------")

predictions = evaluate(best_tree, x_data, y_data)

for i in range(5):
    print("Units Consumed:", x_data[i])
    print("Household Members:", y_data[i])
    print("Actual Bill: Rs.", actual_bills[i])
    print("Predicted Bill: Rs.", round(float(predictions[i]), 2))
    print()


GENE EXPRESSION ALGORITHM
--------------------------------
Best Expression: (((5 + (2 / y)) * x) + 20)
Best Fitness (MSE): 2023.68

ELECTRICITY BILL PREDICTIONS
--------------------------------
Units Consumed: 100
Household Members: 2
Actual Bill: Rs. 640
Predicted Bill: Rs. 620.0

Units Consumed: 150
Household Members: 3
Actual Bill: Rs. 910
Predicted Bill: Rs. 870.0

Units Consumed: 200
Household Members: 4
Actual Bill: Rs. 1180
Predicted Bill: Rs. 1120.0

Units Consumed: 250
Household Members: 3
Actual Bill: Rs. 1410
Predicted Bill: Rs. 1436.67

Units Consumed: 300
Household Members: 5
Actual Bill: Rs. 1700
Predicted Bill: Rs. 1640.0

